# COMP 322 PyTorch Wizard: in-class exercises

This notebook is **not for turning in**. We work through it together in class. Nothing here is collected or graded, so use it to experiment, break things, and ask questions.

> **Attribution.** This notebook adapts and expands material from Daniel Bourke's *Learn PyTorch for Deep Learning*, chapters "00. PyTorch Fundamentals" and "01. PyTorch Workflow Fundamentals" (https://www.learnpytorch.io/). Credit goes to Daniel Bourke for the teaching approach, including the "three big errors" (shape, dtype, device) and the habit of predicting before running. It would be good for you to read both chapters. All code and exercises here are original COMP 322 adaptations written for this course.

**How to work.**

* Work in pairs. **Predict first** (shapes, dtypes, values), then run, then explain any surprise out loud.
* Cells marked `TODO` or raising `NotImplementedError` are the tasks we do together. Assertions check *contracts* (shapes, dtypes, values), not every possible implementation.
* Use vectorized tensor operations unless a task explicitly allows a loop.
* CPU is plenty. Everything here runs in well under a minute. A GPU is only needed for the first section, and only to see it working.
* Everything is seeded with **322**, so numbers should match from run to run.

**Plan for the session.**

| Part | Topic | Rough time |
|---|---|---|
| Setup | Set up a GPU in Colab, the three big errors | 10 min |
| Exercise 1 | Memory-sharing detective | 15 min |
| Exercise 2 | Translation traps (NumPy habit to torch fix) | 20 min |
| Exercise 3 | Channel normalization without loops | 15 min |
| Exercise 4 | Autograd debugging clinic | 20 min |
| Wrap-up | Exit check (discussion only) | 5 min |
| Optional | Use your own GPU locally | on your own time |

## Set up a GPU in Colab

A GPU runs the same tensor code on thousands of small cores at once. For the small examples in this notebook you will not feel a difference, but you should know how to switch one on, because larger models need it. Colab gives you a free GPU with some limits.

**Step by step.**

1. Open the notebook in Colab.
2. In the menu choose **Runtime**, then **Change runtime type**.
3. Under **Hardware accelerator** choose **GPU** (Colab usually offers a T4). Press **Save**.
4. Colab restarts the runtime, so any variables you had are gone. Run the notebook again from the top.
5. Run the verification cell below. It should print `CUDA available: True` and the GPU name.

**About the free tier.** The free GPU is a shared resource with usage limits. A GPU is not guaranteed at every moment, long or idle sessions can be disconnected, and your quota can run out for the day. When you are done, use **Runtime**, then **Disconnect and delete runtime**, so that others can use the GPU. If no GPU is available, everything in this notebook still works on the CPU.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())     # True only on a GPU runtime

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    # Create a tensor DIRECTLY on the GPU (faster than building it on the CPU and copying it).
    x_gpu = torch.ones(3, device="cuda")
    print("x_gpu lives on:", x_gpu.device)              # cuda:0
    # Or move an existing CPU tensor. .to() returns a NEW tensor, so reassign the result.
    y_gpu = torch.tensor([1.0, 2.0, 3.0]).to("cuda")
    print("x_gpu + y_gpu:", x_gpu + y_gpu)              # both operands are on the GPU
    # NumPy cannot see GPU memory: bring the tensor back to the CPU first.
    print("back on the CPU as NumPy:", y_gpu.cpu().numpy())
else:
    print("No GPU runtime. Follow steps 2 to 5 above, then rerun this cell. The rest of the notebook runs on the CPU anyway.")

**Is the GPU actually faster?** Not always. For tiny tensors the cost of launching work on the GPU and copying data dominates. For a big matrix multiplication the GPU wins by a large factor. GPU kernels also run **asynchronously**: Python returns before the GPU has finished, so you must call `torch.cuda.synchronize()` before you stop the clock, or you will time nothing. Run the cell below and compare. (On a CPU-only runtime it just times the CPU.)

In [ ]:
import time

def time_matmul(device, n=2048, repeats=5):
    # Time n x n matrix multiplications on the given device ("cpu" or "cuda").
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    _ = a @ b                                     # warm-up run (GPU start-up costs are paid here)
    if device == "cuda":
        torch.cuda.synchronize()                  # wait until the GPU is idle before starting the clock
    start = time.perf_counter()
    for _ in range(repeats):
        _ = a @ b
    if device == "cuda":
        torch.cuda.synchronize()                  # wait for the GPU to finish before stopping the clock
    return (time.perf_counter() - start) / repeats

cpu_time = time_matmul("cpu")
print(f"CPU: {cpu_time * 1000:.1f} ms per 2048 x 2048 matmul")
if torch.cuda.is_available():
    gpu_time = time_matmul("cuda")
    print(f"GPU: {gpu_time * 1000:.1f} ms per 2048 x 2048 matmul  (about {cpu_time / gpu_time:.0f}x faster)")

## Imports, seed, and device

From here on we use one `device` variable. The pattern is: choose `device` once, then write `.to(device)` on every tensor or model that needs to move. A fuller guide, including Apple and AMD hardware, is the optional last section.

In [ ]:
import math
import warnings

import numpy as np

import torch
from torch import nn

SEED = 322  # the course seed: every random choice below starts from this number


def set_seed(seed=SEED):
    # Seed both libraries so NumPy habits and torch code are reproducible together.
    torch.manual_seed(seed)
    np.random.seed(seed)


set_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"     # see the last section for Apple (mps) and others
print("torch", torch.__version__, "| numpy", np.__version__, "| device", device)

## The three big errors: shape, dtype, device

Bourke points out that three problems cause most PyTorch errors. When something breaks, ask which of the three it is.

| Error | Typical message | Fix |
|---|---|---|
| **Shape** | `mat1 and mat2 shapes cannot be multiplied` | check `.shape`, then `reshape`, `permute`, or transpose |
| **dtype** | `expected ... to have the same dtype` | `.to(torch.float32)` |
| **device** | `Expected all tensors to be on the same device` | `.to(device)` on both operands |

Run the next cell and read each message. Do not skip the messages: learning to read them is most of debugging.

In [ ]:
# 1. Shape: (3, 2) @ (3, 2) has mismatched inner dimensions (2 versus 3).
try:
    torch.rand(3, 2) @ torch.rand(3, 2)
except RuntimeError as err:
    print("shape :", str(err)[:90])

# 2. dtype: float32 @ int64 will not multiply. Convert one of them.
try:
    torch.rand(3) @ torch.tensor([1, 2, 3])
except RuntimeError as err:
    print("dtype :", str(err)[:90])
print("fixed :", (torch.rand(3) @ torch.tensor([1, 2, 3]).to(torch.float32)).shape, "(a scalar tensor)")

# 3. device: only reproducible when a GPU is present.
if device == "cuda":
    try:
        torch.rand(3, device="cuda") + torch.rand(3)           # one operand on the GPU, one on the CPU
    except RuntimeError as err:
        print("device:", str(err)[:90])
else:
    print("device: (needs a GPU runtime to reproduce; the message says 'Expected all tensors to be on the same device')")

## NumPy habits that need revisiting

You already know NumPy. That is both a gift and a trap: PyTorch tensors *look* like NumPy arrays, share many function names, and even share memory with them, but the **defaults differ** in ways that bite. The four exercises below are a field guide to the sharp edges.

**Rule for every exercise here: predict first, run second, explain third.** Record your prediction in a comment or in the provided dictionary *before* executing. A wrong prediction is the most valuable thing that can happen in this notebook.

## Exercise 1: Memory-sharing detective

NumPy users are used to *views* and *copies*. Torch adds two more verbs, `detach` and `clone`, and conversions that sometimes share memory and sometimes do not.

| Call | Shares memory with the source? | Keeps autograd history? |
|---|---|---|
| `torch.from_numpy(arr)` | **yes** (zero copy) | no (new leaf) |
| `torch.tensor(arr)` | no (always copies) | no |
| `torch.as_tensor(arr)` | yes if dtype and device already match | no |
| `t.numpy()` (CPU tensor, no grad) | **yes** | n/a |
| `t.detach()` | **yes** | **no** |
| `t.clone()` | no | **yes** |
| `t.detach().clone()` | no | no |

The table is the answer key. Do not read it again until you have filled the prediction dictionaries below, using only what you already know about tensors.

In [ ]:
array = np.arange(4, dtype=np.float32)
a = torch.from_numpy(array)
b = torch.tensor(array)
c = a.clone()

# TODO (predict BEFORE running the checks): does each tensor share memory with `array`?
predicted_sharing = {"a": None, "b": None, "c": None}   # fill with True / False

array[0] = 42.0        # mutate the NumPy array in place
print("array:", array)
print("a    :", a.numpy())
print("b    :", b.numpy())
print("c    :", c.numpy())

# Measured truth: np.shares_memory compares the underlying buffers.
actual_sharing = {name: bool(np.shares_memory(array, t.numpy())) for name, t in [("a", a), ("b", b), ("c", c)]}
print("predicted:", predicted_sharing, "\nactual   :", actual_sharing)
assert predicted_sharing == actual_sharing, "Your prediction disagreed with reality: reread the table"

In [ ]:
tracked = torch.tensor([2.0, 3.0], requires_grad=True)
detached = tracked.detach()
cloned = tracked.clone()

# TODO (predict): which of the two shares storage with `tracked`? which keeps requires_grad?
shares_storage = {"detach": None, "clone": None}
keeps_grad = {"detach": None, "clone": None}

real_storage = {"detach": detached.data_ptr() == tracked.data_ptr(), "clone": cloned.data_ptr() == tracked.data_ptr()}
real_grad = {"detach": detached.requires_grad, "clone": cloned.requires_grad}
print("shares storage:", real_storage, "| requires_grad:", real_grad)
assert shares_storage == real_storage, "storage prediction wrong"
assert keeps_grad == real_grad, "requires_grad prediction wrong"

**Independent NumPy snapshot.** You want to log the current value of a trainable tensor as a NumPy array, and you want that log entry to stay unchanged while training continues to update the tensor in place. Neither `detach()` alone (shares memory) nor `clone()` alone (still requires grad, so `.numpy()` fails) is enough. Combine them.

In [ ]:
tracked = torch.tensor([2.0, 3.0], requires_grad=True)    # fresh tensor so this cell can be re-run

# TODO: snapshot = an independent NumPy array holding [2.0, 3.0] (before the update below).
snapshot = None  # YOUR CODE HERE

# Simulate an optimizer step: in-place update, outside autograd.
with torch.no_grad():
    tracked.add_(10.0)

assert isinstance(snapshot, np.ndarray), "snapshot must be a NumPy array"
assert snapshot.tolist() == [2.0, 3.0], "snapshot changed when tracked was updated: it still shares memory"
assert tracked.tolist() == [12.0, 13.0]
print("snapshot", snapshot, "| tracked now", tracked.tolist())

**Why sharing is dangerous: the version counter.** Autograd saves some tensors for the backward pass and records a *version number* for them. Mutating a shared buffer through the detached alias bumps the version, and autograd refuses to continue. Read the output.

In [ ]:
w_demo = torch.tensor([1.0, 2.0], requires_grad=True)
loss_demo = (w_demo ** 2).sum()           # pow saves w_demo for backward
alias = w_demo.detach()                   # shares memory AND the version counter
alias.zero_()                             # in-place edit through the alias
try:
    loss_demo.backward()
except RuntimeError as err:
    print("RuntimeError:", str(err)[:140])

**dtype trap.** NumPy defaults to `float64`; torch defaults to `float32`. `nn.Linear` weights are `float32`, so a `float64` input fails with a dtype mismatch. Convert with `.to(torch.float32)` (which **copies** when the dtype changes and returns the *same object* when it does not).

In [ ]:
x64 = torch.from_numpy(np.linspace(0, 1, 5))        # float64, shares memory with the NumPy temp
print("x64 dtype:", x64.dtype)
try:
    nn.Linear(1, 1)(x64.unsqueeze(-1))
except RuntimeError as err:
    print("RuntimeError:", str(err)[:100])

# TODO: x32 = float32 version of x64 using .to(...)
x32 = None  # YOUR CODE HERE

assert x32.dtype == torch.float32
assert x32.data_ptr() != x64.data_ptr(), "a dtype change must allocate new memory"
assert x32.to(torch.float32) is x32, ".to with the same dtype returns the very same tensor (no copy)"
print("Linear on float32 works:", nn.Linear(1, 1)(x32.unsqueeze(-1)).shape)

**Devices are a third reason for copies.** Moving a tensor between devices can allocate new memory, but only when the tensor is not already there. Predict first: what does `.to(device)` return for a tensor that is already on `device`, and what does `.cpu()` return for a tensor that is already on the CPU?

In [ ]:
t = torch.arange(4.0)                      # lives on the CPU

same_device = t.to("cpu")                  # already on the CPU: no copy, the very same object
same_cpu = t.cpu()                         # already on the CPU: also the very same object
print(".to('cpu') is t :", same_device is t)
print(".cpu() is t      :", same_cpu is t)

moved = t.to(device)                       # on a CPU-only runtime this is the same object; on a GPU it is a COPY
print("moved is t       :", moved is t, "| device:", moved.device)

# The rule of thumb: if you need an independent copy, say so explicitly with .clone().
independent = t.clone()
independent[0] = 99.0
print("t unchanged      :", t.tolist())

**Your answer** (when does `.to(device)` copy, and when does it return the same object?): _write here_

**Your answer** (3 to 5 sentences; double-click to edit): _write here_

## Exercise 2: Translation traps (NumPy habit → torch fix)

Each cell below shows a NumPy habit that fails, or silently does the wrong thing, in PyTorch. For each one: **predict**, run the demo, write the fix, and say whether your fix **copies**, changes **dtype**, or changes **device**.

In [ ]:
# --- Trap 1: standard deviation defaults disagree ---------------------------------------------
values = torch.tensor([1.0, 2.0, 3.0, 4.0])
np_std = float(np.std(values.numpy()))     # NumPy default: ddof=0 (population)
torch_std = values.std().item()            # torch default: correction=1 (sample / Bessel)
print("np.std = %.4f | torch.std = %.4f" % (np_std, torch_std))   # predict which is larger first!

# TODO: make torch match NumPy, and NumPy match torch. Read the `correction` / `ddof` arguments.
torch_matching_numpy = None   # YOUR CODE HERE (a Python float from a torch call)
numpy_matching_torch = None   # YOUR CODE HERE (a Python float from a NumPy call)

assert math.isclose(torch_matching_numpy, np_std, rel_tol=1e-6), "torch.std needs correction=0"
assert math.isclose(numpy_matching_torch, torch_std, rel_tol=1e-6), "np.std needs ddof=1"
print("aligned: both libraries give %.4f (population) and %.4f (sample)" % (np_std, torch_std))

In [ ]:
# --- Trap 2: negative-step slices do not exist in torch ----------------------------------------
vec = torch.arange(5)
try:
    vec[::-1]
except ValueError as err:
    print("ValueError:", err)

# TODO: reverse `vec` with torch.flip (note dims must be a tuple or list).
reversed_vec = None  # YOUR CODE HERE

assert reversed_vec.tolist() == [4, 3, 2, 1, 0]
# Unlike NumPy's [::-1] (a view), torch.flip COPIES. Confirm by checking the memory addresses.
assert reversed_vec.data_ptr() != vec.data_ptr()

# --- Trap 3: astype -> .to ------------------------------------------------------------------------
ints = torch.tensor([1, 2, 3, 4])           # int64

# --- Trap 4: mean of integers --------------------------------------------------------------------
try:
    ints.mean()
except RuntimeError as err:
    print("RuntimeError:", str(err)[:90])
# TODO: the mean of ints as a Python float (2.5), the way np.mean(ints) gives it. Convert dtype first.
int_mean = None  # YOUR CODE HERE
assert int_mean == 2.5
print("fixed mean:", int_mean)

In [ ]:
# --- Trap 5: .numpy() on a tensor that requires grad (or lives on a GPU) -------------------------
w = torch.tensor([1.0, 2.0], requires_grad=True)
try:
    w.numpy()
except RuntimeError as err:
    print("RuntimeError:", str(err)[:100])


def to_numpy(t):
    # TODO: a conversion that is safe for ANY tensor: tracked by autograd, on a GPU, or both.
    #   Order matters: detach from the graph, move to CPU, then convert.
    raise NotImplementedError("implement to_numpy")


non_leaf = w * 2                               # has grad_fn: the hardest case
arr = to_numpy(non_leaf)
assert isinstance(arr, np.ndarray) and arr.tolist() == [2.0, 4.0]
print("to_numpy ok. (On a CUDA tensor the .cpu() step is what makes it work; here it is a no-op.)")

# --- Trap 6: np.max returns a value; torch.max(dim=...) returns (values, indices) -----------------
M = torch.tensor([[1.0, 9.0, 3.0], [7.0, 2.0, 5.0]])
# TODO: row_max (values only, shape (2,)) and row_argmax (indices only, shape (2,)).
row_max = None     # YOUR CODE HERE
row_argmax = None  # YOUR CODE HERE
assert row_max.tolist() == [9.0, 7.0] and row_argmax.tolist() == [1, 0]

In [ ]:
# --- Trap 7: transpose versus permute, and views versus copies ------------------------------------
# NumPy's np.transpose(a, axes) takes a full axis order. torch.transpose(a, d0, d1) swaps only TWO dims.
nhwc = torch.arange(2 * 4 * 5 * 3).reshape(2, 4, 5, 3)       # (N, H, W, C)

wrong = nhwc.transpose(0, 3)                                  # swaps N and C only!
print("transpose(0, 3) shape:", tuple(wrong.shape), "(not NCHW!)")

# TODO: correct NHWC -> NCHW reorder of ALL axes in one call.
nchw = None  # YOUR CODE HERE
assert nchw.shape == (2, 3, 4, 5)
assert nchw[1, 2, 3, 4] == nhwc[1, 3, 4, 2], "element (n, c, h, w) must equal nhwc[n, h, w, c]"

# permute returns a VIEW with permuted strides: it is not contiguous in memory anymore.
print("nchw contiguous?", nchw.is_contiguous())
try:
    nchw.view(2, -1)                                          # view needs compatible strides
except RuntimeError as err:
    print("view -> RuntimeError:", str(err)[:80])

# TODO: flatten each example to (2, 60) in a way that works on a non-contiguous tensor.
#   Two valid answers: .reshape(...) or .contiguous().view(...). Which one copies?
flat = None  # YOUR CODE HERE
assert flat.shape == (2, 60)

**Your answer** (one sentence per trap; double-click to edit): _write here_

## Exercise 3: Channel normalization without loops (NHWC → NCHW)

Image libraries (Pillow, OpenCV, NumPy images from Lab 4) store images as **NHWC**: (batch, height, width, channels). PyTorch convolutions expect **NCHW**. A very common preprocessing step is **per-image, per-channel standardization**:

```
out[n, c] = (x[n, c] - mean over (H, W) of x[n, c]) / max(std over (H, W) of x[n, c], 1e-6)
```

Requirements:

* Convert NHWC → NCHW with one `permute`.
* Compute the mean and **population** standard deviation (`correction=0`) over the spatial axes `(-2, -1)` with `keepdim=True`, so broadcasting lines up.
* **Do not mix examples or channels.**
* Clamp the std to at least `1e-6` so a **constant image** gives finite output (all zeros) instead of `nan`.
* A batch of size one must stay a batch: shape `(1, 3, H, W)`.
* No loops in the function. (The test cell uses a loop-based reference; that is allowed in a test.)

In [ ]:
images = torch.arange(120, dtype=torch.float32).reshape(2, 4, 5, 3)    # deterministic NHWC batch


def normalize_images(nhwc):
    # TODO: 1) nchw = permute NHWC -> NCHW
    #       2) mean over dims (-2, -1), keepdim=True      -> (N, C, 1, 1)
    #       3) std over dims (-2, -1), correction=0, keepdim=True, then .clamp(min=1e-6)
    #       4) return (nchw - mean) / std
    raise NotImplementedError("implement normalize_images")


result = normalize_images(images)
assert result.shape == (2, 3, 4, 5)
torch.testing.assert_close(result.mean(dim=(-2, -1)), torch.zeros(2, 3), atol=1e-5, rtol=0)
torch.testing.assert_close(result.std(dim=(-2, -1), correction=0), torch.ones(2, 3), atol=1e-4, rtol=0)

# Reference implementation with loops (only used to check you; loops are fine in a test).
reference = torch.empty(2, 3, 4, 5)
for n in range(2):
    for c in range(3):
        channel = images[n, :, :, c]                              # (H, W)
        reference[n, c] = (channel - channel.mean()) / max(channel.std(correction=0).item(), 1e-6)
torch.testing.assert_close(result, reference, atol=1e-5, rtol=1e-5)

# Independence: changing example 1 must not change example 0; changing channel 0 must not change channel 1.
perturbed = images.clone()
perturbed[1] = perturbed[1] * 7 + 3
torch.testing.assert_close(normalize_images(perturbed)[0], result[0])
perturbed = images.clone()
perturbed[..., 0] = perturbed[..., 0].flip(0)
torch.testing.assert_close(normalize_images(perturbed)[:, 1:], result[:, 1:])

assert normalize_images(images[:1]).shape == (1, 3, 4, 5), "batch of one must keep its batch axis"
constant = normalize_images(torch.ones(1, 4, 5, 3))
assert torch.isfinite(constant).all() and constant.abs().max() == 0, "constant images should map to zeros"
print("All normalization checks passed.")

**Your answer** (2 to 4 sentences; double-click to edit): _write here_

## Exercise 4: Autograd debugging clinic

Six classic bugs, then the derivative of a cubic computed three ways. For each bug the cell **shows the failure**; you write the repair and the assertion proves it.

Remember the three modes you should be able to explain without notes:

* `model.eval()` changes the behavior of *layers* (dropout off, batch norm uses running averages). It does **not** turn autograd off.
* `torch.no_grad()` disables graph recording for the operations inside it; tensors created there are ordinary tensors that simply have no history.
* `torch.inference_mode()` is a stricter, faster `no_grad` for pure evaluation: tensors created inside it can never be used in autograd, ever.

**Bug A: a NumPy operation inside the loss.** NumPy knows nothing about autograd, so the graph is cut the moment you round-trip through it.

In [ ]:
pred = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)


def buggy_loss_a(pred):
    # BUG: leaves torch for NumPy and rebuilds a brand-new tensor with no history.
    return torch.tensor(np.square(pred.detach().numpy()).sum())


bad = buggy_loss_a(pred)
print("buggy loss requires_grad:", bad.requires_grad, "| grad_fn:", bad.grad_fn)
try:
    bad.backward()
except RuntimeError as err:
    print("RuntimeError:", str(err)[:90])


def fixed_loss_a(pred):
    # TODO: the same sum of squares using ONLY torch operations on `pred`.
    raise NotImplementedError("fix bug A")


pred.grad = None
fixed_loss_a(pred).backward()
torch.testing.assert_close(pred.grad, 2 * pred.detach())     # d/dp sum(p^2) = 2p
print("Bug A fixed: grad =", pred.grad.tolist())

**Bug B: `torch.tensor(existing_tensor)` inside `forward`.** It copies the data into a new leaf and silently disconnects the parameter from the loss. (PyTorch warns you; do not ignore warnings.)

In [ ]:
class BuggyScale(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.tensor(2.0))

    def forward(self, x):
        # BUG: torch.tensor(...) of a tensor makes a NEW leaf with no connection to self.w.
        return torch.tensor(self.w * x)


class FixedScale(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.tensor(2.0))

    def forward(self, x):
        # TODO: scale x by self.w while staying connected to the graph.
        raise NotImplementedError("fix bug B")


xs = torch.tensor([1.0, 2.0, 3.0])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    out_bad = BuggyScale()(xs)
print("buggy output requires_grad:", out_bad.requires_grad, "| warnings raised:", len(caught))

fixed = FixedScale()
fixed(xs).sum().backward()
assert fixed.w.grad is not None and fixed.w.grad.item() == 6.0, "d/dw sum(w * x) = sum(x) = 6"
print("Bug B fixed: w.grad =", fixed.w.grad.item())

**Bug C: missing `zero_grad`.** `backward()` **accumulates** into `.grad`. That is a feature (gradient accumulation over micro-batches) and a bug when forgotten.

In [ ]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
for step in range(3):
    loss = (w ** 2).sum()
    loss.backward()                  # BUG: never clears w.grad, so the three gradients are added up
print("accumulated grad:", w.grad.tolist(), "(expected 2w =", (2 * w).tolist(), ")")


def grad_without_accumulation(w, steps=3):
    # TODO: repeat the same forward/backward `steps` times but make sure the final w.grad equals ONE
    # gradient, not `steps` of them. Return a detached copy of w.grad.
    raise NotImplementedError("fix bug C")


w = torch.tensor([1.0, 2.0], requires_grad=True)
torch.testing.assert_close(grad_without_accumulation(w), torch.tensor([2.0, 4.0]))
print("Bug C fixed. (Deliberate accumulation: divide each micro-batch loss by the number of micro-batches.)")

**Bug D: `backward()` after computing the loss inside `inference_mode`.** Tensors created in `inference_mode` carry no graph, so there is nothing to backpropagate.

In [ ]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
x = torch.tensor([3.0, 4.0])

with torch.inference_mode():
    loss = (w * x).sum()             # BUG: training computation placed inside inference_mode
print("loss requires_grad:", loss.requires_grad)
try:
    loss.backward()
except RuntimeError as err:
    print("RuntimeError:", str(err)[:90])


def fixed_loss_d(w, x):
    # TODO: compute the loss in normal (grad-enabled) mode. inference_mode is for EVALUATION only.
    raise NotImplementedError("fix bug D")


w.grad = None
l = fixed_loss_d(w, x)
assert l.requires_grad
l.backward()
torch.testing.assert_close(w.grad, x)                 # d/dw sum(w * x) = x
with torch.inference_mode():                          # the legitimate use: measuring, not training
    eval_value = (w * x).sum().item()
print("Bug D fixed. Evaluation under inference_mode gives", eval_value)

**Bug E: believing `eval()` disables autograd.** Predict, then measure.

In [ ]:
net = nn.Sequential(nn.Linear(2, 2), nn.Dropout(0.5))
net.eval()                                      # layer behavior only
inp = torch.ones(1, 2)

# TODO: compute out_eval = net(inp) in eval mode; record whether it requires grad.
eval_requires_grad = None
# TODO: compute the same output under torch.no_grad(); record whether it requires grad.
no_grad_requires_grad = None

assert net.training is False
assert eval_requires_grad is True, "eval() does not disable autograd"
assert no_grad_requires_grad is False, "no_grad is what disables the graph"
print("eval():", eval_requires_grad, "| no_grad():", no_grad_requires_grad)

**Bug F: keeping graph-connected losses in a list.** Appending the loss *tensor* keeps its whole graph (and every saved activation) alive: a memory leak that grows each epoch.

In [ ]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
leaky = []
for i in range(3):
    leaky.append((w * i).sum())                 # BUG: each entry keeps its autograd graph alive
print("leaky entries still have grad_fn:", [t.grad_fn is not None for t in leaky])


def collect_losses(w, n=3):
    # TODO: same loop, but store plain Python floats so the graphs can be freed.
    raise NotImplementedError("fix bug F")


history = collect_losses(w)
assert all(isinstance(h, float) for h in history), "store floats via .item()"
assert history == [0.0, 3.0, 6.0]
print("Bug F fixed:", history)

### The cubic: `backward` vs `autograd.grad` vs `create_graph`

Let `f(x) = sum(x^3)` at `x = [-2, 0, 3]`. Then `df/dx = 3 x^2 = [12, 0, 27]` and `d²f/dx² = 6 x = [-12, 0, 18]`.

1. **`f.backward()`** *accumulates* into `x.grad`. Use it for training.
2. **`torch.autograd.grad(f, x)`** *returns* the gradient as a tuple and leaves `x.grad` untouched. Use it when you want a derivative as a value, for example inside a loss that contains a gradient penalty.
3. For a **second derivative**, the first gradient must itself be differentiable: pass `create_graph=True`.

Build a **fresh graph** for each approach. Do not use `retain_graph=True` as a blanket fix: it keeps the graph in memory and hides the real problem (reusing a freed graph).

In [ ]:
x0 = torch.tensor([-2.0, 0.0, 3.0])

# --- 1) backward: populates .grad ---------------------------------------------------------------
x1 = x0.clone().requires_grad_(True)
f1 = None                  # TODO: sum of x1 cubed
# TODO: call backward on f1
grad_backward = None       # TODO: x1.grad (a snapshot copy)

# --- 2) autograd.grad: returns the gradient, does NOT touch .grad -----------------------------------
x2 = x0.clone().requires_grad_(True)       # fresh leaf, fresh graph
f2 = None                  # TODO: sum of x2 cubed (a NEW forward pass)
(grad_autograd,) = [None]  # TODO: torch.autograd.grad(f2, x2) returns a TUPLE; unpack it

# --- 3) second derivative with create_graph=True ----------------------------------------------------
x3 = x0.clone().requires_grad_(True)
f3 = None                  # TODO: sum of x3 cubed
first = None               # TODO: torch.autograd.grad(f3, x3, create_graph=True)[0]  (differentiable!)
second = None              # TODO: differentiate first.sum() with respect to x3 -> should be 6 * x3

assert grad_backward is not None, "TODO: fill in the three approaches"
expected = torch.tensor([12.0, 0.0, 27.0])
torch.testing.assert_close(grad_backward, expected)
torch.testing.assert_close(grad_autograd, expected)
assert x2.grad is None, "autograd.grad must not populate x2.grad"
torch.testing.assert_close(second, 6 * x0)
print("first derivative:", grad_backward.tolist(), "| second derivative:", second.tolist())

**Your answer** (one line per bug, plus the API difference; double-click to edit): _write here_

---
# Wrap-up: exit check (discussion only)

Nothing to submit. Before we finish, each pair should be able to say the following out loud:

1. Which of `from_numpy`, `torch.tensor`, `detach`, `clone`, `.numpy()`, and `.to(device)` share memory with their source, and which do not?
2. Name three defaults that differ between NumPy and torch (for example `std`, slicing with a negative step, `max` with an axis) and the torch fix for each.
3. Why does `permute`, and not `transpose(0, 3)`, convert NHWC to NCHW?
4. Why must `.cpu()` come before `.numpy()` on a GPU tensor, and why must `detach()` come before it for a tensor that requires grad?
5. What is the difference between `eval()`, `no_grad()`, and `inference_mode()`?

---
# Optional: use your own GPU locally

This last section is **optional**. Colab is enough for this course. If you want to use the GPU or accelerator in your own computer, the idea is always the same: install a PyTorch build that matches your hardware, check that PyTorch sees it, then let one `device` variable carry the choice through your code.

| Hardware | Install source | PyTorch device name | How to check |
|---|---|---|---|
| **NVIDIA** GPU (Windows or Linux) | CUDA build from pytorch.org | `cuda` | `nvidia-smi`, `torch.cuda.is_available()` |
| **AMD** GPU (mainly Linux) | ROCm build from pytorch.org | `cuda` (yes, the same name) | `rocm-smi`, `torch.version.hip` |
| **Apple silicon** Mac | default macOS install | `mps` | `torch.backends.mps.is_available()` |
| **Intel** GPU | XPU build, see the PyTorch docs | `xpu` | `torch.xpu.is_available()` |
| none of the above | default CPU build | `cpu` | always works |

## NVIDIA (CUDA)

1. **Check the driver.** Open a terminal and run `nvidia-smi`. If it prints a table with your GPU name and a driver version, the driver is installed. If the command is not found, install the current NVIDIA driver for your card first.
2. **Install PyTorch.** Go to https://pytorch.org/get-started/locally/, choose your OS, **Pip** (or Conda), **Python**, and a **CUDA** version. The page shows the exact command, which looks like this (the CUDA tag in the URL changes over time, so copy it from the page):

```
pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126
```

3. You do **not** need to install the CUDA toolkit separately: the PyTorch wheels bundle the CUDA runtime. You only need a driver that is new enough.
4. Verify with the cell below.

## AMD (ROCm)

ROCm support is strongest on Linux with supported Radeon and Instinct cards. On pytorch.org choose the **ROCm** compute platform and run the command it shows. ROCm builds of PyTorch deliberately reuse the `cuda` name, so `torch.cuda.is_available()` is `True` and `device = "cuda"` works unchanged. To tell them apart, `torch.version.hip` is a version string on a ROCm build and `None` on an NVIDIA build. Check your exact card against the ROCm compatibility list first, because not every consumer GPU is supported.

## Apple silicon (MPS)

On a Mac with an M-series chip the normal `pip install torch` already includes **MPS** (Metal Performance Shaders) support. Use `torch.device("mps")`. Two practical notes: some operations are not implemented on MPS yet (set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` **before** importing torch to let those fall back to the CPU), and MPS does not support `float64`, so keep your tensors in `float32`.

## Check what PyTorch sees

In [ ]:
import shutil
import subprocess

# nvidia-smi is a separate program; it exists only if an NVIDIA driver is installed.
if shutil.which("nvidia-smi"):
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True)
    if out.returncode == 0:
        print("nvidia-smi:", out.stdout.strip())
    else:
        print("nvidia-smi was found but could not talk to a driver:", out.stdout.strip()[:80])
else:
    print("nvidia-smi: not found (no NVIDIA driver on this machine)")

print("torch version         :", torch.__version__)          # a '+cpu' suffix means the CPU-only build is installed
print("built with CUDA       :", torch.version.cuda)         # None on CPU-only, AMD, and Apple builds
print("built with ROCm (HIP) :", torch.version.hip)          # a version string on AMD builds
print("cuda available        :", torch.cuda.is_available())  # True for NVIDIA and AMD
print("number of GPUs        :", torch.cuda.device_count())
print("mps available         :", torch.backends.mps.is_available())     # Apple silicon
print("xpu available         :", hasattr(torch, "xpu") and torch.xpu.is_available())   # Intel GPUs

## The device-agnostic pattern

Write the choice **once**, in one function, and never hard-code `"cuda"` elsewhere. The same script then runs on a laptop CPU, an NVIDIA workstation, an AMD box, or a Mac.

In [ ]:
def pick_device():
    # Return the best available device, in order of preference.
    if torch.cuda.is_available():                       # NVIDIA CUDA, and AMD ROCm builds
        return torch.device("cuda")
    if torch.backends.mps.is_available():               # Apple silicon
        return torch.device("mps")
    if hasattr(torch, "xpu") and torch.xpu.is_available():   # Intel GPU
        return torch.device("xpu")
    return torch.device("cpu")                          # always works


dev = pick_device()
print("chosen device:", dev)

# The whole pattern in a few lines: model and data both move with .to(dev).
set_seed()
model = nn.Linear(3, 1).to(dev)                         # the model's parameters now live on dev
xb = torch.randn(8, 3).to(dev)                          # the batch must live on the SAME device
yb = (xb @ torch.tensor([[1.0], [2.0], [3.0]], device=dev))   # new tensors can be created directly on dev
loss = nn.functional.mse_loss(model(xb), yb)
loss.backward()                                         # gradients are computed on dev too
print("loss", round(loss.item(), 3), "| weight grad device:", model.weight.grad.device)   # .item() copies a scalar to the CPU

# Going back to NumPy: detach from autograd, move to the CPU, then convert (the order matters).
print("as NumPy:", model.weight.detach().cpu().numpy())

**Habits that save time.**

* Create tensors **on** the device (`torch.zeros(3, device=dev)`) instead of building on the CPU and copying.
* Save checkpoints with `torch.save(model.state_dict(), path)` and load them with `torch.load(path, map_location=dev, weights_only=True)`, so a file written on a GPU loads anywhere.
* To compare runs, seed the GPU generator too: `torch.manual_seed` also seeds CUDA, and `torch.cuda.manual_seed_all(seed)` exists for several GPUs. Exact GPU reproducibility can still need `torch.use_deterministic_algorithms(True)`.
* When timing GPU code, call `torch.cuda.synchronize()` before reading the clock (see the Colab section).

**If `cuda available` says `False` although you own a GPU.** (1) The version string ends in `+cpu`: you installed the CPU build, so reinstall using the command from pytorch.org. (2) `nvidia-smi` fails: the driver is missing or too old. (3) You are in a different Python environment from the one where you installed PyTorch: check `import sys; print(sys.executable)`. (4) On a laptop with two GPUs the program may be running on the integrated one: select the discrete GPU in your system settings.